# 06 · Cobertura y monto de vías de protección económica, ENASEM 2001–2021

**Pregunta de tesis propuesta por la autora:** ¿qué tanto es consistente la expansión de pensiones/apoyos con reducción de brechas de cobertura/monto, quién permanece sin vía propia identificada y cómo cambia la lectura al considerar muerte y no observación?

La idea 'se iguala acceso pero no monto/suficiencia' es hipótesis por verificar. Este análisis describe muestra, no cobertura nacional ni efecto de políticas. Mortalidad y selección son sensibilidades del eje económico.

**Objetivo independiente:** unión propia K/A y tipología completa de personas en 2018/2021, brechas dentro de ronda, distribución nominal por evento y suma parcial por persona, persistencia de negativas K en ventanas comparables. Reutiliza integración/titularidad y lógica 05; no modifica notebooks 01–05 ni bases. No panel balanceado.

**Elección de la autora:** principal solo directas; respuestas válidas por sustituto como sensibilidad. Población 60+ viva por visita, con N excluido por modalidad reportado. A admite dinero/especie; cobertura identificada no garantiza ingreso monetario.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown, Image
from tesis_enasem.economic_protection import analyze, OUTPUT, verify_inputs, GROUPS
from tesis_enasem.economic_protection_outputs import wave, conclusions
from tesis_enasem.project_paths import RESULTS_DIR, PROJECT_ROOT
verify_inputs()
r = analyze()  # reproduce solo06 desde bases verificadas; no rehace01–05
stored = json.loads((OUTPUT/'resultados.json').read_text())
assert r == stored, 'Agregados06 no corresponden al código/insumos actuales'
figuras = json.loads((OUTPUT/'figuras.json').read_text())
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
pd.set_option('display.float_format',lambda x:f'{x:.3f}')
def tabla(rows):
    display(pd.json_normalize(rows,sep='.'))
def figura(key):
    f=next(x for x in figuras if x['clave']==key)
    display(Image(filename=str(RESULTS_DIR/f['archivo'])))
    display(Markdown('**Nota:** '+f['nota']))
def principal():return [z for z in r['rondas'] if z['analisis']=='principal']


## 1. Fuentes, nombres, códigos y poblaciones

Maestro 99,676/27,171 intacto. ID cunicah:np, persona+ronda única; LEFT 1:1. Edad por visita, escolaridad al inicio, localidad a 100 mil. Ocupación 4 agrega retiro/incapacidad/no trabajo. est_conyugal no acredita pareja residente. imp_neto es stock compartido y no se usa como ingreso.

In [ ]:
contrato=json.loads((PROJECT_ROOT/'diccionarios/contrato_proteccion_economica.json').read_text()); tabla([r['metodologia']]); tabla(r['contratos_originales']); tabla([{'variable':k,'definicion':v} for k,v in contrato['variables'].items()]); tabla([{'ronda':z['ronda'],**z['cobertura'][0],'N_proxy_K_original_conocida':z['N_proxy_K_original_conocida']} for z in principal()])

**Interpretación y límites:** N 60=56,004 observaciones/18,754 personas; sumas entre rondas no son individuos distintos. Sustitutos no se borran: recepción válida fuera del principal. Elegibilidad operativa exige 60+ viva/directa; no certifica elegibilidad legal.

## 2. Comparabilidad antes de comparar

K condicionada 2001–03,2012–15,2018–21; A condicionada 2003–12,2012–15,2015–18. Unión exige ambas: solo 2012–15 cumple continuidad condicional conjunta. No extrapolar nombres de programas/monto/modalidad.

In [ ]:
tabla(r['comparabilidad'])

**Interpretación y límites:** Brechas dentro de ronda describen ese instrumento. Cambios de brechas entre rupturas también pueden reflejar medición diferencial/composición; no quedan homologados por restar porcentajes. A 2021 incorpora vejez explícita; no atribuir salto exclusivamente a política.

## 3. Tipología: conteos generales 2018/2021

Ocho combinaciones con pareja confirmada, cuatro propias K/A sin pareja residente, desconocidos de contexto/recepción y sustitutos aparte. Se agregan cónyuge+A y las tres vías, omitidas inicialmente. Solo A es propia, no A del cónyuge.

In [ ]:
tabla([{'ronda':z['ronda'],**a} for z in principal() for a in z['tipologia'] if a['dimension']=='General']); figura('tipologia')

**Interpretación y límites:** Cada observación pertenece a una categoría/contexto. No pareja residente no es pensión negativa imputada. Negativas K/A sin pareja no son 'ninguna de tres' por defecto. Ninguna vía identificada no equivale a falta de protección integral. Unión conocida puede tener tipologíaND si falta alguna respuesta.

## 4. Tipología por sexo, edad, escolaridad, localidad y ocupación

N conocido, NND, Nsinpareja y proporción del grupo/contexto; distribución general y por perfiles.

In [ ]:
for dimension in GROUPS:
    display(Markdown('### '+dimension))
    df=pd.DataFrame([{'ronda':z['ronda'],**a} for z in principal() for a in z['tipologia'] if a['dimension']==dimension])
    display(df.pivot_table(index=['grupo','contexto','categoria'],columns='ronda',values='N',fill_value=0))
    tabla(df[['ronda','grupo','contexto','N_grupo','N_contexto','N_conocido_contexto','N_ND_contexto','N_sin_pareja_grupo']].drop_duplicates().to_dict('records'))

**Interpretación y límites:** Cero en conteos significa cero personas de esa categoría, nunca respuesta faltante sustituida. No juntar contextos como supuesta pensión de hogar.

## 5. Cobertura propia y brechas en seis rondas

K∨A: sí si alguna sí; no solo ambas no; restoND. Tasas usan distintos N conocidos; contraste K frente a unión exige ambas conocidas, mismo soporte.

In [ ]:
for dimension in ['General',*GROUPS]:
    display(Markdown('### '+dimension))
    tabla([{'ronda':z['ronda'],**a} for z in principal() for a in z['cobertura'] if a['dimension']==dimension])
figura('cobertura2021'); figura('brechas')

**Interpretación y límites:** ND no es negativa. Límites entre elegibles son extremos lógicos, no IC. Sin pesos/diseño no inferencia nacional. A puede ser especie. Diferencias 2021 sexo/localidad se atenúan con unión, pero edad se separa más.

## 6. Brechas y sensibilidad por modalidad

Todas las comparaciones dirigidas de pares, no selección por mayor brecha. Principal frente a respuestas válidas por sustituto con cada N conocido.

In [ ]:
tabla([{'ronda':z['ronda'],**a} for z in principal() for a in z['brechas']]); tabla([{'ronda':z['ronda'],'analisis':z['analisis'],**a} for z in r['rondas'] for a in z['cobertura'] if a['dimension'] in ['General','sexo_grupo','localidad_grupo']])

**Interpretación y límites:** Contraste de modalidades puede reflejar composición/selección. No se corrigen directas con sustitutos. Tipologías y cobertura completa de sensibilidad en r['rondas'].

## 7. Montos: evento frente a persona y vía

Originales observados de mes normal. Varios eventos por receptor; suma parcial personal no es mediana por evento ni ingreso total. Cotejo exacto sidecar, no reemplazo. No distribuir instituciones, sumar K+A ni imputados.33 firmas 2015 siguen sin corregir.

In [ ]:
tabla([{'via':k,**v} for k,v in r['auditoria_eventos'].items()]); tabla([a for a in r['montos'] if a['dimension']=='General']); figura('montos2021')

**Interpretación y límites:** Nreceptores/personas con monto/todos eventos capturados con monto/parciales/eventosND separados. Todos capturados no prueba todos los beneficios reales. Cero observado se conserva; faltante no es cero. A dinero/especie no separable.

## 8. Montos por perfil y tipología

Cuantiles/medias nominales por vía dentro de ronda; todas las rondas y perfiles en agregados,2018/2021 se muestran detallados.

In [ ]:
for dimension in GROUPS:
    display(Markdown('### '+dimension))
    tabla([a for a in r['montos'] if a['dimension']==dimension and a['ronda'] in [2018,2021]])
tabla([a for a in r['montos'] if a['dimension']=='grupo_tipologia']); tabla(r['montos_conyuge_contextuales'])

**Interpretación y límites:** Cónyuge se presenta como monto contextual de otra persona con pareja confirmada, no ingreso propio/transferencia comprobada. Mediana A común no implica distribución igual o suficiencia. Comparaciones entre años nominales, no reales.

## 9. Deflactación/suficiencia: preparada, no ejecutada

Confirmar INPC (fuente/base/frecuencia) y umbral mensual compatible con año/unidad/localidad. evaluate_sufficiency exige fuentes confirmadas y cobertura de índices/umbrales, sin valores supuestos.

In [ ]:
from tesis_enasem.economic_protection import evaluate_sufficiency
config=contrato['config_suficiencia']
assert not config['fuente_indice_confirmada']
assert not r['metodologia']['deflactacion_ejecutada']
display(Markdown('**Estado: bloqueada por fuentes y umbral pendientes.**'))
tabla([config])

**Interpretación y límites:** No deflactación ni proporción suficiente/insuficiente ejecutadas. Línea de pobreza requiere localidad/unidad; salario diario conversión mensual documentada. Comparar prestación parcial con referencia no diagnostica pobreza del hogar.

## 10. Seguimiento de negativa K

Basal 60+, negativa K conocida directa.2012–15/2018–21 principales;2001–03 sensibilidad. Destino conserva edad discordante. No panel balanceado; negativa K no excluye A u otros recursos.

In [ ]:
tabla([{'intervalo':f"{z['origen']}–{z['destino']}",'N_cohorte':z['N_cohorte_60'],'N_directa_basal':z['N_elegible_basal'],'meses_N':z['meses_N'],'meses_mediana':z['meses_mediana'],'meses_min':z['meses_min'],'meses_max':z['meses_max'],**z['grupos'][0]} for z in r['seguimiento_principal']]); figura('seguimiento')

**Interpretación y límites:** Estados disjuntos: negativa viva,K observada,muerte registrada,ausencia,vivo KND/modalidad yotro no clasificable. Sustituto no es ausencia. Registro sobre fallecido no fecha/mortalidad completa. Inconsistencia fallecido→vivo del maestro conservada sin reasignar.

## 11. Persistencia, cotas y cobertura seleccionada

Negativa entre sobrevivientes conocidos y negativa viva por cohorte basal son estimandos distintos. Cotas no asignan recepción a fallecidos; extremos entre ausentes/medicionesND. Mortalidad registrada D/N a(D+U)/N.

In [ ]:
tabla([{'intervalo':f"{z['origen']}–{z['destino']}",**a} for z in r['seguimiento_principal'] for a in z['grupos']]); tabla([{'intervalo':f"{z['origen']}–{z['destino']}",'cohorte':z['cobertura_basal_cohorte'],'retenidos':z['cobertura_basal_retenidos']} for z in r['seguimiento_principal']]); tabla([{'intervalo':f"{z['origen']}–{z['destino']}",**z['grupos'][0]} for z in r['seguimiento_sensibilidad_sustitutos']])

**Interpretación y límites:** Reaparición viva y registro fallecido posterior separados; pueden solaparse por anomalías, no fechan estado en visita ausente.2021 sin búsqueda posterior dentro de estas rondas. Cotas no son IC ni correcciones. Solo 2012–15 lectura conjunta K/A condicionada.

## 12. Síntesis y siguiente decisión

Conservar contradicciones, montos faltantes y rupturas. Entender cobertura antes de precios/umbral; predicción opcional requiere estimando/etiquetas defendibles.

In [ ]:
display(Markdown('\n\n'.join(conclusions(r)))); verify_inputs()

**Interpretación y límites:** No modelos opcionales, KM/Cox/IPW, política causal, alerta temprana oQ 1. Validar medición/diseño/fuentes; no convertir sobrevivientes en población inicial ni imputar prestaciones a fallecidos.

## 13. Reproducción y productos

`python scripts/run_economic_protection.py`; `python scripts/build_economic_protection_notebook.py`; `python scripts/validate_economic_protection.py`; `python scripts/export_economic_protection_report.py`; `python scripts/build_report_pdf.py`; `python scripts/build_report_pdf.py --check`.

Agregados completos en `data/interim/cobertura_proteccion_economica/resultados.json`, figuras en `resultados/figuras/`, contrato/configuración en `diccionarios/contrato_proteccion_economica.json`. Fuente sin outputs; copia ejecutada en `data/interim/notebooks_ejecutados/`. Único informe acumulativo, sin informes nuevos. Bases, notebooks 01–05 y manuscrito intactos; sin Git/GitHub.